In [66]:
# Activar modo gráfico
try:
    %matplotlib widget
except Exception:
    %matplotlib inline
    print("Interactividad deshabilitada: Modo 'inline'.")

# --- Librerías ---
import folium
import geopandas as gpd
import ipywidgets as widgets
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from IPython.display import clear_output, display

Interactividad deshabilitada: Modo 'inline'.


In [ ]:
# --- Datos ---
df = pd.read_csv(
    "https://raw.githubusercontent.com/dosquisd/AccidentalRiskAnalysis/refs/heads/main/data/processed/dataset.csv"
)
df["FECHA_OCURRENCIA"] = pd.to_datetime(df["FECHA_OCURRENCIA"])

df.head()

,OBJECTID,FORMULARIO,CODIGO_ACCIDENTE,FECHA_OCURRENCIA,DIA_SEMANA_OCURRENCIA,GRAVEDAD,CLASE,LOCALIDAD,latitude,longitude
0,646545,A00410124,306211,2008-09-05 12:50:00,5,CON HERIDOS,ATROPELLO,TEUSAQUILLO,4.655402,-74.101700
1,563782,839933000,291565,2008-05-20 23:40:00,2,SOLO DANOS,CHOQUE,ENGATIVA,4.689417,-74.109351
2,4398952,A000038718,4398952,2014-12-04 17:30:00,4,SOLO DANOS,CHOQUE,ENGATIVA,4.689750,-74.109016
3,1612,726750500,30243,2007-06-29 19:30:00,5,CON HERIDOS,ATROPELLO,PUENTE ARANDA,4.615129,-74.108947
4,569692,840564600,297475,2008-07-21 07:30:00,1,SOLO DANOS,CHOQUE,TEUSAQUILLO,4.646696,-74.108792


In [71]:
# --- Widgets ---
# Localities dropdown, control interactivo de localidad
localities_col = sorted(df["LOCALIDAD"].unique().tolist())
locality_dropdown = widgets.Dropdown(
    options=["ALL"] + localities_col,
    description="Localidad:",
    disabled=False,
)

# Years slider, control interactivo de años
years = sorted(df["FECHA_OCURRENCIA"].dt.year.unique())
year_slider = widgets.IntRangeSlider(
    value=[min(years), max(years)],
    min=min(years),
    max=max(years),
    step=1,
    description="Años:",
    disabled=False,
    continuous_update=False,
    orientation="horizontal",
    readout=True,
    readout_format="d",
)

out = widgets.Output()


# Función dinámica que actualiza los lineplots
def update_lineplot(locality, year_range):
    start_year, end_year = year_range

    # Filtramos por año
    mask = (df["FECHA_OCURRENCIA"].dt.year >= start_year) & (
        df["FECHA_OCURRENCIA"].dt.year <= end_year
    )
    df_filtered = df[mask]

    # Filtramos por localidad si no es 'ALL'
    if locality != "ALL":
        df_filtered = df_filtered[df_filtered["LOCALIDAD"] == locality]
    with out:
        clear_output(wait=True)
        # Agrupamos por localidad y resample semanal
        localities_col = df_filtered["LOCALIDAD"].unique().tolist()
        localities = {}
        for loc in localities_col:
            localities[loc] = (
                df_filtered[df_filtered["LOCALIDAD"] == loc]
                .set_index("FECHA_OCURRENCIA")
                .resample("1W")
                .size()
            )

        if not localities:
            print("No hay datos para la selección actual.")
            return

        # Escala y gráfico
        locality_max = max(localities, key=lambda x: localities[x].max())
        scale = localities[locality_max].max()

        fig, ax = plt.subplots(figsize=(10, 5))
        for loc, series in localities.items():
            sns.lineplot(data=series, ax=ax, label=loc)

        ax.set(
            title=f"Semanal Accidents ({start_year}-{end_year})",
            xlabel="Date",
            ylabel="Number of Accidents",
            ylim=(0, scale + 5),
        )
        ax.grid()
        ax.legend(bbox_to_anchor=(1, 1), loc="upper left")
        plt.show()


# Dashboard interactivo
widgets.interact(
    update_lineplot, locality=locality_dropdown, year_range=year_slider
)
display(out)

interactive(children=(Dropdown(description='Localidad:', options=('ALL', 'ANTONIO NARIÑO', 'BARRIOS UNIDOS', '…

Output()

In [ ]:
!pip install folium geopandas

In [67]:
# Configuración inicial
folium_colors = [
    "red",
    "blue",
    "green",
    "purple",
    "orange",
    "darkred",
    "lightred",
    "beige",
    "darkblue",
    "darkgreen",
    "cadetblue",
    "darkpurple",
    "pink",
    "lightblue",
    "lightgreen",
    "gray",
    "black",
    "lightgray",
]

# Cargar GeoDataFrame
gdf = gpd.GeoDataFrame(
    pd.read_csv(
        "https://raw.githubusercontent.com/dosquisd/AccidentalRiskAnalysis/refs/heads/main/data/processed/dataset.csv"
    ),
    geometry=gpd.points_from_xy(df["longitude"], df["latitude"]),
    crs="EPSG:4326",  # sistema geográfico WGS84 (Lat/Lon)
)

localities_cols = list(gdf["LOCALIDAD"].unique())

# Polígonos de las localidades
localities_gdf = gpd.read_file("/content/poligonos-localidades.zip").to_crs(
    "EPSG:4326"
)

# Drop unnecesary columns and rename
localities_gdf = localities_gdf[
    localities_gdf["Nombre_de_l"].isin(localities_cols)
]

localities_gdf.drop(
    columns=["Acto_admini", "Area_de_la_", "Identificad"], inplace=True
)
localities_gdf.rename(columns={"Nombre_de_l": "LOCALIDAD"}, inplace=True)

display(localities_gdf)

,LOCALIDAD,geometry
0,SANTA FE,"POLYGON ((-73.99446 4.61425, -73.99446 4.61425..."
1,BARRIOS UNIDOS,"POLYGON ((-74.05725 4.68684, -74.06249 4.65594..."
2,FONTIBON,"POLYGON ((-74.10342 4.65351, -74.1075 4.64823,..."
3,ENGATIVA,"POLYGON ((-74.15547 4.71798, -74.15547 4.71798..."
4,CANDELARIA,"POLYGON ((-74.06621 4.60317, -74.0662 4.60317,..."
5,TUNJUELITO,"POLYGON ((-74.13777 4.59489, -74.13165 4.59363..."
6,PUENTE ARANDA,"POLYGON ((-74.1183 4.63741, -74.11504 4.64053,..."
7,RAFAEL URIBE URIBE,"POLYGON ((-74.12803 4.59254, -74.12777 4.59233..."
8,KENNEDY,"POLYGON ((-74.1183 4.63741, -74.11845 4.63727,..."
9,USME,"POLYGON ((-74.05597 4.50832, -74.05611 4.50822..."


In [73]:
df["year"] = df["FECHA_OCURRENCIA"].dt.year

metric_dropdown = widgets.Dropdown(
    options=["Total de accidentes", "Con heridos", "Solo daños", "Con Muertos"],
    value="Total de accidentes",
    description="Métrica:",
)

out = widgets.Output()


# Función  dinámica para actualizar el mapa
def update_map(year_range, metric):
    start_year, end_year = year_range
    df_filtered = df[
        (df["FECHA_OCURRENCIA"].dt.year >= start_year)
        & (df["FECHA_OCURRENCIA"].dt.year <= end_year)
    ]
    with out:
        clear_output(wait=True)

        # Calcular métrica seleccionada
        if metric == "Total de accidentes":
            accidents_by_loc = (
                df_filtered.groupby("LOCALIDAD")
                .size()
                .reset_index(name="ACCIDENTES")
            )
        elif metric == "Con heridos":
            accidents_by_loc = (
                df_filtered[
                    df_filtered["GRAVEDAD"].str.contains(
                        "HERIDOS", case=False, na=False
                    )
                ]
                .groupby("LOCALIDAD")
                .size()
                .reset_index(name="ACCIDENTES")
            )
        elif metric == "Solo daños":
            accidents_by_loc = (
                df_filtered[
                    df_filtered["GRAVEDAD"].str.contains(
                        "DANOS", case=False, na=False
                    )
                ]
                .groupby("LOCALIDAD")
                .size()
                .reset_index(name="ACCIDENTES")
            )
        elif metric == "Con Muertos":
            accidents_by_loc = (
                df_filtered[
                    df_filtered["GRAVEDAD"].str.contains(
                        "MUERTOS", case=False, na=False
                    )
                ]
                .groupby("LOCALIDAD")
                .size()
                .reset_index(name="ACCIDENTES")
            )

        # Unir con polígonos
        merged = localities_gdf.merge(
            accidents_by_loc, on="LOCALIDAD", how="left"
        )
        merged["ACCIDENTES"] = merged["ACCIDENTES"].fillna(0)

        # Crear mapa
        bogota_map = folium.Map(
            location=[4.6485784, -74.1031911],
            zoom_start=11,
            tiles="openstreetmap",
        )

        # Añadir capa de accidentes
        choropleth = folium.Choropleth(
            geo_data=merged,
            data=merged,
            columns=["LOCALIDAD", "ACCIDENTES"],
            key_on="feature.properties.LOCALIDAD",
            fill_color="YlOrRd",
            fill_opacity=0.9,
            line_opacity=0.5,
            legend_name=f"{metric} ({start_year}-{end_year})",
        ).add_to(bogota_map)

        # Añadir tooltips
        folium.GeoJsonTooltip(
            fields=["LOCALIDAD", "ACCIDENTES"],
            aliases=["Localidad:", "Accidentes:"],
            localize=True,
        ).add_to(choropleth.geojson)

        return bogota_map


# Mostrar dashboard
widgets.interact(update_map, year_range=year_slider, metric=metric_dropdown)
display(out)

interactive(children=(IntRangeSlider(value=(2007, 2013), continuous_update=False, description='Años:', max=201…

Output()